<h1> Hands on: SCF with <span style="color: red;">QE</span>py</h1>

#### Run SCF loop and get converged densitites and energies



## To run this tutorial we need to
 - import QEpy
 - Use ASE to generate molecular structures

### To run this tutorial in google colab click the following link :

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asesma-org/miniASESMA2026/blob/main/Week1/Day3/hands-on-scf/hands-on_scf.ipynb)


In [2]:
## Uncomment the following lines for Colab. IMPORTANT NOTE: install first qepy and then dftpy
!pip install qepy f90wrap==0.2.16

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.9/113.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 103.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires n

In [ ]:
# !python -m pip install "git+https://github.com/Quantum-MultiScale/DFTpy.git@dev"

In [ ]:
import qepy                        # QEpy related modules
from qepy.driver import Driver     #
from qepy.io import QEInput        #

from ase.build import molecule     # ASE molecule builder

import numpy as np                 # numpy!

import matplotlib.pyplot as plt    # Plotting helpers: matplotlib and plotly
import plotly.io as pio            #
pio.renderers.default = 'iframe'   #
import plotly.graph_objects as go  #

## Download some pseudopotentials

In [ ]:
additional_files = {
    'H.pz-vbc.UPF' : 'https://pseudopotentials.quantum-espresso.org/upf_files/H.pz-vbc.UPF',
    'C.pz-vbc.UPF' : 'https://pseudopotentials.quantum-espresso.org/upf_files/C.pz-vbc.UPF',
}
from dftpy.formats import download_files
download_files(additional_files)

## Generate the QE options to run a scf calculation

In [ ]:
prefix = 'c2h4' # this is needed by any QE / QEpy calculation

In [ ]:
scf_options = {}

scf_options['&control'] = {}
scf_options['&control']['calculation'] = 'scf'
scf_options['&control']['prefix'] = prefix
scf_options['&control']['restart_mode'] = 'from_scratch'
scf_options['&control']['outdir'] = 'tmp'
scf_options['&control']['pseudo_dir'] = './'

scf_options['&system'] = {}
scf_options['&system']['ecutwfc'] = 25 # the PW cutoff in Ry

scf_options['&electrons'] = {}
scf_options['&electrons']['conv_thr'] = 1.0E-6  # SCF convergence threshold

scf_options['atomic_species'] = []
scf_options['atomic_species'].append('H  1.008  H.pz-vbc.UPF')
scf_options['atomic_species'].append('C 12.011  C.pz-vbc.UPF')

scf_options['k_points {gamma}'] = []

## Workflow for SCF

 1. Use ASE to build a new molecule
 2. Set periodic cell
 3. Update  QEpy options dictionary
 4. Initialize QEpy `Driver` class
 5. Run SCF
 6. Save wavefunction and eigenvalues data - and plot them!

## Steps 1-3: Build molecule, set cell and generate options dictionary

In [ ]:
atoms = molecule ("C2H4")                # define the molecule
atoms.set_cell(cell=np.identity(3)*10)   # set simulation cell (in Angstroms)
atoms.translate([5,5,5])                 # translate molecule to center of cell (for convenience)
pwin = QEInput()                         # input handler of QEpy
scf_options = pwin.update_atoms(atoms, qe_options=scf_options)  # initialize the QEpy input

<br>
<center><b>What does the input look like?</b></center>

In [ ]:
from pprint import pprint
pprint(scf_options)

{'&control': {'calculation': 'scf',
              'outdir': 'tmp',
              'prefix': 'c2h4',
              'pseudo_dir': './',
              'restart_mode': 'from_scratch'},
 '&electrons': {'conv_thr': 1e-06},
 '&system': {'ecutwfc': 25, 'ibrav': 0, 'nat': 6, 'ntyp': 2},
 'atomic_positions angstrom': ['C    5.00000000000000 5.00000000000000 '
                               '5.66748000000000',
                               'C    5.00000000000000 5.00000000000000 '
                               '4.33252000000000',
                               'H    5.00000000000000 5.92283200000000 '
                               '6.23769500000000',
                               'H    5.00000000000000 4.07716800000000 '
                               '6.23769500000000',
                               'H    5.00000000000000 5.92283200000000 '
                               '3.76230500000000',
                               'H    5.00000000000000 4.07716800000000 '
                             

## Update QEpy options some more

 - `nbnd` needs to be large enough (molecule dependent)

In [ ]:
scf_options["&system"]["nbnd"] = 10

## Step 4: Initialize the Driver class with the scf QE options

In [ ]:
driver = Driver(qe_options=scf_options, task='scf', logfile=prefix+'.scf.out')

## Step 5: Run a scf calculation

In [ ]:
driver.scf()

-26.85622489714924

This is the total energy after SCF in Hartree.

## Step 6: Save the wavefunctions and all QE data for any follow-up simulations

In [ ]:
driver.save()

## Step 7: Plot KS orbital #5

In [ ]:
qe_field = driver.get_wave_function(band=4,kpt=0)[0].real

In [ ]:
dftpy_field = driver.data2field(qe_field)

In [ ]:
## Uncomment the following line for Colab
pio.renderers.default = "colab"

X,Y,Z = dftpy_field.grid.r[0],dftpy_field.grid.r[1],dftpy_field.grid.r[2]

fig = go.Figure(data=go.Isosurface(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=dftpy_field.flatten(),
    isomin=-2, isomax=2, # The range of values to show
    surface_count=2,          # Number of "layers"
    opacity=0.9,
    caps=dict(x_show=False, y_show=False) # Hides the box sides
))
fig.show()

### You can also save it to an `xsf` file for VESTA

In [ ]:
dftpy_field.write(ions=driver.get_dftpy_ions(),filename="dftpy_field.xsf")

# Challenge (s)

Plot other interesting things:
 - Plot the electron density using `driver.get_density()`
 - Plot anoth KS orbital
 - Plot occupations (`driver.get_occupation_numbers()`) vs. orbital energies (`driver.get_eigenvalues()`)
 - Do so for another molecule of your choice.